In [ ]:

# ============================================================
# 1. Load dataset
# ============================================================
print("Loading dataset...")

df = pd.read_csv('/content/mds_ed.csv')

print(f"Dataset shape: {df.shape}")

Loading dataset...
Dataset shape: (451, 1936)


['general_ecg_time', 'general_ecg_no_within_stay']

['vitals_heartrate_mean',
 'vitals_heartrate_median',
 'vitals_heartrate_min',
 'vitals_heartrate_max',
 'vitals_heartrate_std',
 'vitals_heartrate_first',
 'vitals_heartrate_last',
 'vitals_heartrate_rate_change',
 'vitals_heartrate_coeff']

array([0, 1])

In [10]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import warnings

warnings.filterwarnings("ignore")

# ----------------------------------------
# 1. Load dataset
# ----------------------------------------
print("Loading data...")
df = pd.read_csv("/content/mds_ed.csv")
print("Shape:", df.shape)




# vasopressors(agent that raises blood pressure by constricting blood vessels)
vasopressors_columns = [i for i in df.columns if 'vasopressors' in i] # it is a binary class whether the patient has vasopressors or no
Heartrate_columns = [i for i in df.columns if 'heart' in i] # checking if missing any other features (included in vitals)
Bloodpressure_cols= [i for i in df.columns if 'blood' in i] # included in lab_values

ECG_cols = [i for i in df.columns if 'ecg_' in i]
ECG_cols # for ECG data there is only when an ECG was taken and which ECG it is within a hospital stay

df['deterioration_vasopressors'].unique()

# ----------------------------------------
# 2. Feature groups
# ----------------------------------------
demographics_columns = [c for c in df.columns if "demographics_" in c]
biometrics_columns   = [c for c in df.columns if "biometrics_" in c]
vitals_columns       = [c for c in df.columns if "vitals_" in c]
labvalues_columns    = [c for c in df.columns if "labvalues_" in c]

features = (
    demographics_columns +
    biometrics_columns +
    vitals_columns +
    labvalues_columns
)

print("Number of features:", len(features))

# ----------------------------------------
# 3. Target: vasopressors
# ----------------------------------------
target_col = "deterioration_vasopressors"

if target_col not in df.columns:
    raise ValueError(f"{target_col} not found in dataset")

# Keep only rows where label exists
df = df.dropna(subset=[target_col])

# Convert to integer (important for XGBoost)
df[target_col] = df[target_col].astype(int)

# ----------------------------------------
# 4. Train-set median imputation (replacing missing vls with the mean)
# ----------------------------------------
train_mask = df["general_strat_fold"].isin(range(0, 18))
medians = df.loc[train_mask, features].median()
df[features] = df[features].fillna(medians)

# ----------------------------------------
# 5. Train / Val / Test split (data split into 20 folds one for testing, one for val, rest for training )
# ----------------------------------------
train_df = df[df["general_strat_fold"].isin(range(0, 18))].reset_index(drop=True)
val_df   = df[df["general_strat_fold"] == 18].reset_index(drop=True)
test_df  = df[df["general_strat_fold"] == 19].reset_index(drop=True)

# Use first ECG only (same as your setup)
val_df  = val_df[val_df["general_ecg_no_within_stay"] == 0]
test_df = test_df[test_df["general_ecg_no_within_stay"] == 0]

x_train = train_df[features].values
x_val   = val_df[features].values
x_test  = test_df[features].values

y_train = train_df[target_col].values
y_val   = val_df[target_col].values
y_test  = test_df[target_col].values

print("\nShapes:")
print("Train:", x_train.shape, "Val:", x_val.shape, "Test:", x_test.shape)

# ----------------------------------------
# 6. Train XGBoost model
# ----------------------------------------
model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    n_jobs=4
)

model.fit(
    x_train, y_train,
    eval_set=[(x_val, y_val)],
    verbose=False
)

# ----------------------------------------
# 7. Evaluate
# ----------------------------------------
y_pred = model.predict_proba(x_test)[:, 1]
auroc = roc_auc_score(y_test, y_pred)

print("\n===== RESULT =====")
print("vasopressors AUROC:", round(auroc, 4))
print("Positive rate:", y_test.mean())

Loading data...
Shape: (129057, 1936)
Number of features: 470

Shapes:
Train: (116433, 470) Val: (5824, 470) Test: (6080, 470)

===== RESULT =====
VASOPRESSORS AUROC: 0.9159
Positive rate: 0.008881578947368421
